# Dual Teacher ABSA — huấn luyện trên Kaggle

Notebook này chạy toàn bộ pipeline gồm 3 giai đoạn (`train_asqp_mt5.py`,
`train.py` + `teacher/`, `models/`, `utils/` ở gốc repo):

1. **(Nếu cần) Fine-tune ASQP cho mT5** — `hotel-mt5/` (từ `dapt/train_dapt.py`)
   chỉ mới domain-adapted qua denoising, CHƯA biết sinh quad. Nếu bạn chưa có
   một bản `hotel-mt5-asqp/` (đã fine-tune ASQP) sẵn, notebook sẽ tự
   fine-tune từ `hotel-mt5/` trên nhãn ASQP trước.
2. Train **Extractive Teacher** (XLM-R + Span/Relation/Classification head) trên
   nhãn ASQP (`data_final/labeled_data/.../{train,val}.json`).
3. Chạy cả hai teacher (Generative `hotel-mt5-asqp` + Extractive vừa train) trên
   review chưa gán nhãn, đối chiếu qua **Architectural Disagreement**, gộp điểm
   tin cậy qua **Confidence Fusion**, xuất `pseudo_labels.json`.

## Trước khi chạy

1. Vào **Settings** → bật **GPU** (T4 x2 hoặc P100) và **Internet** (cần để tải
   `xlm-roberta-base` từ HuggingFace Hub, và `pip install`).
2. **Add Input** các dataset Kaggle chứa (tuỳ bạn đã upload sẵn ở đâu — có thể
   dùng thẳng Output của notebook DAPT trước đó, như danh sách file bạn đã
   thấy trong dialog Add Input: `hotel-mt5/`, `checkpoints/checkpoint-*/`, ...):
   - Nhãn ASQP: các file `train.json` / `val.json` / `test.json` (từ
     `data_final/labeled_data/hamos26/`).
   - Review chưa gán nhãn: `hotel_review_merged.csv` (từ `data_final/unlabeled_data/`).
   - Backbone DAPT đã domain-adapt: thư mục tên đúng là `hotel-mt5` (có
     `config.json` + `model.safetensors` + tokenizer) — dùng làm điểm khởi đầu
     để fine-tune ASQP.
   - *(tuỳ chọn, để bỏ qua bước 1)* Nếu bạn đã từng chạy notebook này và tải
     lại `hotel-mt5-asqp.zip` làm Input, thư mục tên đúng `hotel-mt5-asqp` sẽ
     được dùng thẳng, không fine-tune lại.

Cell bên dưới tự dò các đường dẫn này dưới `/kaggle/input` (đệ quy qua symlink),
**ưu tiên khớp đúng tên thư mục** (`hotel-mt5`, `hotel-mt5-asqp`) thay vì chỉ dò
theo tên file `config.json`/`model.safetensors` — vì các checkpoint DAPT trung
gian (`checkpoints/checkpoint-4000/`, `-4500/`, `-5000/`) cũng có 2 file đó và
dễ bị nhận nhầm.


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GiB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))


In [ ]:
# Bật GPU và Internet trong Settings trước khi chạy
!git clone https://github.com/hotuyen21pt/MultiLABSA.git

# Kaggle đã có sẵn torch/transformers/pandas/numpy/tqdm — chỉ cài thêm
# sentencepiece + protobuf (cần cho tokenizer mT5). KHÔNG cài lại torch để
# tránh phá bản CUDA của Kaggle.
!pip install -q -r MultiLABSA/requirements-kaggle.txt


In [ ]:
import os, fnmatch

# Kaggle mount dataset dưới /kaggle/input bằng symlink tới thư mục phiên bản
# (vd /kaggle/input/<slug>/versions/N/...), nên phải os.walk(followlinks=True)
# để duyệt xuyên qua symlink — mặc định (followlinks=False) sẽ báo "không tìm
# thấy file" dù dataset đã được Add Input đúng cách.

# Tra ve thu muc dau tien duoi root co TEN (basename) khop mot trong "names".
# Dung cho hotel-mt5 / hotel-mt5-asqp de tranh nham voi checkpoints/checkpoint-N/
# (chung cung co config.json + model.safetensors nhung KHONG phai backbone dung).
def find_dir_by_name(names, root="/kaggle/input"):
    for dirpath, _dirs, _files in os.walk(root, followlinks=True):
        if os.path.basename(dirpath.rstrip("/\\")) in names:
            return dirpath
    return None

# Tra ve thu muc dau tien duoi root chua DU tat ca filenames.
def find_dir_containing(filenames, root="/kaggle/input"):
    for dirpath, _dirs, files in os.walk(root, followlinks=True):
        if all(f in files for f in filenames):
            return dirpath
    return None

# Tra ve duong dan file dau tien khop bat ky pattern nao.
def find_file(patterns, root="/kaggle/input"):
    for dirpath, _dirs, files in os.walk(root, followlinks=True):
        for fn in files:
            if any(fnmatch.fnmatch(fn, p) for p in patterns):
                return os.path.join(dirpath, fn)
    return None

LABELED_DIR = find_dir_containing(["train.json", "val.json"])
UNLABELED_CSV = find_file(["hotel_review_merged.csv", "hotel_review*_lang.csv"])
DAPT_BACKBONE_DIR = find_dir_by_name({"hotel-mt5", "hotel_mt5"})
ASQP_MODEL_DIR = find_dir_by_name({"hotel-mt5-asqp", "hotel_mt5_asqp", "hotel-mt5_asqp"})

print("LABELED_DIR       :", LABELED_DIR or "KHONG TIM THAY - can Add Input dataset chua train.json/val.json")
print("UNLABELED_CSV     :", UNLABELED_CSV or "KHONG TIM THAY - can Add Input dataset chua hotel_review_merged.csv")
print("DAPT_BACKBONE_DIR :", DAPT_BACKBONE_DIR or "khong co - se can neu ASQP_MODEL_DIR cung khong co")
print("ASQP_MODEL_DIR    :", ASQP_MODEL_DIR or "khong co - se fine-tune moi tu DAPT_BACKBONE_DIR")

assert LABELED_DIR, "Thieu nhan ASQP (train.json/val.json) duoi /kaggle/input."
assert UNLABELED_CSV, "Thieu review chua gan nhan (hotel_review_merged.csv) duoi /kaggle/input."
assert ASQP_MODEL_DIR or DAPT_BACKBONE_DIR, (
    "Khong co ca hotel-mt5-asqp lan hotel-mt5 duoi /kaggle/input - "
    "khong the chay Generative Teacher. Van co the tiep tuc voi --skip_pseudo_labeling "
    "neu chi muon train Extractive Teacher."
)


In [ ]:
%cd /kaggle/working/MultiLABSA

if ASQP_MODEL_DIR:
    GEN_MODEL_DIR = ASQP_MODEL_DIR
    print("Da co san hotel-mt5-asqp, bo qua fine-tune:", GEN_MODEL_DIR)
elif DAPT_BACKBONE_DIR:
    GEN_MODEL_DIR = "/kaggle/working/hotel-mt5-asqp"
    print("Fine-tune ASQP tu backbone DAPT:", DAPT_BACKBONE_DIR, "->", GEN_MODEL_DIR)
else:
    GEN_MODEL_DIR = None
    print("Khong co backbone nao - bo qua giai doan Generative Teacher.")


In [ ]:
# Stage 1: fine-tune ASQP cho mT5 (chi chay neu can - xem cell tren)
if ASQP_MODEL_DIR is None and DAPT_BACKBONE_DIR is not None:
    !python train_asqp_mt5.py \
        --labeled_dir {LABELED_DIR} \
        --base_model {DAPT_BACKBONE_DIR} \
        --num_epochs 8 --train_batch_size 8 --eval_batch_size 16 \
        --learning_rate 3e-4 --label_smoothing 0.1 \
        --output_dir /kaggle/working/checkpoints/hotel-mt5-asqp \
        --final_dir /kaggle/working/hotel-mt5-asqp
else:
    print("Bo qua Stage 1 (da co san hoac khong co backbone de fine-tune).")


In [ ]:
# Stage 2 + 3: train Extractive Teacher, roi chay dual-teacher inference + fusion
extra_args = f"--generative_model {GEN_MODEL_DIR}" if GEN_MODEL_DIR else "--skip_pseudo_labeling"

%cd /kaggle/working/MultiLABSA
!python train.py \
    --labeled_dir {LABELED_DIR} \
    --unlabeled_csv {UNLABELED_CSV} \
    --extractive_backbone xlm-roberta-base \
    --train_batch_size 8 --eval_batch_size 16 \
    --num_epochs 5 --learning_rate 2e-5 \
    --relation_threshold 0.5 \
    --alpha 0.4 --beta 0.4 --gamma 0.2 --final_score_threshold 0.5 \
    --output_dir /kaggle/working/checkpoints/dual-teacher \
    --pseudo_labels_out /kaggle/working/checkpoints/dual-teacher/pseudo_labels.json \
    {extra_args}


In [ ]:
import json, os

pseudo_path = "/kaggle/working/checkpoints/dual-teacher/pseudo_labels.json"
if os.path.exists(pseudo_path):
    with open(pseudo_path, encoding="utf-8") as f:
        pseudo = json.load(f)
    total_quads = sum(len(r["quads"]) for r in pseudo)
    print(f"So review co pseudo-label: {len(pseudo)} | tong so quad giu lai: {total_quads}")
    for item in pseudo[:3]:
        print(json.dumps(item, ensure_ascii=False, indent=2))
else:
    print("Khong co pseudo_labels.json (da chay voi --skip_pseudo_labeling hoac chua tim thay generative model).")


In [ ]:
import os, shutil

shutil.make_archive("/kaggle/working/dual-teacher-output", "zip", "/kaggle/working/checkpoints/dual-teacher")
print("Saved /kaggle/working/dual-teacher-output.zip")

# Neu vua fine-tune ASQP o Stage 1, zip lai de lan sau Add Input thang vao
# (khoi phai fine-tune lai) - dat ten thu muc dung la hotel-mt5-asqp.
asqp_dir = "/kaggle/working/hotel-mt5-asqp"
if os.path.isdir(asqp_dir):
    shutil.make_archive("/kaggle/working/hotel-mt5-asqp", "zip", asqp_dir)
    print("Saved /kaggle/working/hotel-mt5-asqp.zip")
